# Análise de dados TCP-CII

### Importação dos parâmetros universais

In [77]:
import importlib.util
from pathlib import Path

path = Path("../../../parametros/config.py").resolve()

spec = importlib.util.spec_from_file_location("parametros", path)
parametros = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parametros)

In [78]:
# Parâmetros importados do arquivo config.py
print(
    "Filtrar por quantidade de alelos TCC1:.........................",
    parametros.filtarar_por_qte_de_alelos_tcc1,
)
print(
    "Parâmetro de filtragem median binding percentile TCC1:.........",
    parametros.parametro_de_filtragem_mbp_tcc1,
)
print(
    "Percentual de match mínimo TCC1:...............................",
    parametros.percent_match_minimo_tcc1,
)

Filtrar por quantidade de alelos TCC1:......................... 10
Parâmetro de filtragem median binding percentile TCC1:......... 1
Percentual de match mínimo TCC1:............................... 95.0


In [79]:
import pandas as pd

In [80]:
df = pd.read_csv('./T CELL/DENV 4 - T Cell Prediction - Class I.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,ASGKLVTQW,303,311,9,HLA-B*57:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993994,0.01
1,1,ASGKLVTQW,303,311,9,HLA-B*58:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993016,0.01
2,1,ITNELNYVLW,71,80,10,HLA-B*57:01,415,0.01,ITNELNYLW,ITNELNYVLW,0.989226,0.01
3,1,SQMLIPKSY,239,247,9,HLA-B*15:01,239,0.01,SQMLIPKSY,SQMLIPKSY,0.980178,0.01
4,1,IESSKNQTW,202,210,9,HLA-B*44:02,202,0.01,IESSKNQTW,IESSKNQTW,0.978499,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
36985,1,LSEKEENMVKSQ,338,349,12,HLA-A*23:01,1367,100.00,LEENMVKSQ,LSEKEENMVKSQ,0.000000,100.00
36986,1,LSEKEENMVKSQ,338,349,12,HLA-A*24:02,1367,100.00,LEENMVKSQ,LSEKEENMVKSQ,0.000000,100.00
36987,1,LSEKEENMVKSQ,338,349,12,HLA-A*32:01,1367,100.00,LSEKEVKSQ,LSEKEENMVKSQ,0.000000,100.00
36988,1,EKEENMVKSQVT,340,351,12,HLA-A*11:01,1369,100.00,ENMVKSQVT,EKEENMVKSQVT,0.000000,100.00


## Selecionando Epítopos por median binding percentile.

In [81]:
mbp_minimo = parametros.parametro_de_filtragem_mbp_tcc1

In [82]:
df_mbp_m5 = df[df['median binding percentile'] < mbp_minimo].copy()
print("Filtrando por median binding percentile < ", mbp_minimo)
df_mbp_m5

Filtrando por median binding percentile <  1


,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,ASGKLVTQW,303,311,9,HLA-B*57:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993994,0.01
1,1,ASGKLVTQW,303,311,9,HLA-B*58:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993016,0.01
2,1,ITNELNYVLW,71,80,10,HLA-B*57:01,415,0.01,ITNELNYLW,ITNELNYVLW,0.989226,0.01
3,1,SQMLIPKSY,239,247,9,HLA-B*15:01,239,0.01,SQMLIPKSY,SQMLIPKSY,0.980178,0.01
4,1,IESSKNQTW,202,210,9,HLA-B*44:02,202,0.01,IESSKNQTW,IESSKNQTW,0.978499,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
650,1,LWPKTHTLW,224,232,9,HLA-A*32:01,224,0.98,LWPKTHTLW,LWPKTHTLW,0.071381,0.98
651,1,MEIRPLSEK,333,341,9,HLA-B*44:02,333,0.98,MEIRPLSEK,MEIRPLSEK,0.058615,0.98
652,1,LWPKTHTLW,224,232,9,HLA-B*57:01,224,0.99,LWPKTHTLW,LWPKTHTLW,0.277500,0.99
653,1,SLIEVKTCLWPK,216,227,12,HLA-A*03:01,1245,0.99,SLIEVKTPK,SLIEVKTCLWPK,0.184120,0.99


## Agrupando por pepitideos e agregando colunas pertinentes

In [83]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    )
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AGPFSQHNY,248,256,1,0.360,HLA-A*30:02
1,AGPFSQHNYR,248,257,2,0.695,"HLA-A*31:01, HLA-A*33:01"
2,ALTPPVSDL,103,111,3,0.330,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06"
3,ALTPPVSDLK,103,112,2,0.100,"HLA-A*03:01, HLA-A*11:01"
4,ALTPPVSDLKY,103,113,6,0.540,"HLA-A*01:01, HLA-A*03:01, HLA-A*26:01, HLA-A*3..."
...,...,...,...,...,...,...
237,YATQTVGPW,260,268,4,0.365,"HLA-B*35:01, HLA-B*53:01, HLA-B*57:01, HLA-B*5..."
238,YGFGMFTTNIW,158,168,2,0.445,"HLA-B*57:01, HLA-B*58:01"
239,YKFQPESPAR,32,41,1,0.570,HLA-A*33:01
240,YVLWEGGHDL,77,86,3,0.230,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06"


## Filtragem por qte_de_alelos

In [84]:
# mbp_minimo = 1
qte_alelos_minimo = parametros.filtarar_por_qte_de_alelos_tcc1

In [85]:
# Filtro do número de alelos
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= qte_alelos_minimo
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,FVVDNVHTW,20,28,14,0.235,"HLA-A*02:06, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,HTWTEQYKF,26,34,10,0.280,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."


## Sorting por median_biding_percentile

In [86]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,FVVDNVHTW,20,28,14,0.235,"HLA-A*02:06, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,HTWTEQYKF,26,34,10,0.280,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [87]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos_tcell_1.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0    FVVDNVHTW
1    HTWTEQYKF
Name: peptide, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [88]:
# !seqkit grep -s -v -r -p '[-*]' './Fastas/denv1_NS1_proteinas.fa' > DENV1_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [89]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_1.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,FVVDNVHTW,9,73.83% (110/149),77.78%,100.00%,NaN
1,2,NP 2,HTWTEQYKF,9,98.66% (147/149),88.89%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [90]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# MPB
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "median_binding_percentile"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# start
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "start"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# end
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "end"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos
0,NP 1,FVVDNVHTW,9,73.83% (110/149),77.78%,100.00%,14,0.235,20,28,"HLA-A*02:06, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,NP 2,HTWTEQYKF,9,98.66% (147/149),88.89%,100.00%,10,0.280,26,34,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [91]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos,percent_match
0,NP 1,FVVDNVHTW,9,73.83% (110/149),77.78%,100.00%,14,0.235,20,28,"HLA-A*02:06, HLA-A*23:01, HLA-A*24:02, HLA-A*2...",73.83
1,NP 2,HTWTEQYKF,9,98.66% (147/149),88.89%,100.00%,10,0.280,26,34,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2...",98.66


### Sort e filtragem por percent_match e presença em alelos

In [92]:
minimum_percent_match = parametros.percent_match_minimo_tcc1

In [93]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= minimum_percent_match]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos,percent_match
0,NP 2,HTWTEQYKF,9,98.66% (147/149),88.89%,100.00%,10,0.28,26,34,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2...",98.66


### Estrutura secundária

In [94]:
def processar_horiz(caminho_horiz):
    """
    Lê o arquivo .horiz do PSIPRED e retorna um dicionário 
    contendo as strings de confiança ('conf'), predição ('pred') e sequência ('seq').
    """
    confianca = ""
    predicao = ""
    sequencia = ""
    
    with open(caminho_horiz, 'r') as f:
        for linha in f:
            linha_str = linha.strip()
            if linha_str.startswith("Conf:"):
                confianca += linha_str.split(":", 1)[1].strip()
            elif linha_str.startswith("Pred:"):
                predicao += linha_str.split(":", 1)[1].strip()
            elif linha_str.startswith("AA:"):
                sequencia += linha_str.split(":", 1)[1].strip()
                
    return {
        "conf": confianca,
        "pred": predicao,
        "seq": sequencia
    }

In [95]:
denv_horiz = processar_horiz("./PSIPRED/denv.horiz")
denv_horiz

{'conf': '9033886999928806068997561331443157059999999999999350907996397999999987438898998735856899988700057324002379954478674225842401866587179981899445994346773389888265365006998867887544783664478209978984264689864899189999999641625478422578788540014469778699553314899530147895248189988446999799579889998632125158962463421679999869996996151425760784434310502239',
 'pred': 'CEECEEECCCCEEECCEEEEEECCCCCCEEECEEECCCHHHHHHHHHHHHHHCCEEEEECCHHHHHHHHHHHHHHHHHHHHCCCCEEEEECCCEEEEECCEECCCCCCCCCCCCCCCCCCCCEECCCCCCCEEEECCCCCCCCCHHHEEECEEEEEEEEEEEECEEEEEEECCCCCCCCCHHHHHHHHCCCEEEEECCCEEEEECCCCEEEEEEEEEEEEEECCCCCCCCCCCCCCCCCCCCCCHHHCCCCCCCCCCCCCCCCCCCCCCCCCEEEEEEECCCCEEEECCCCCCCCCCCCEEECCCCCCCCEECCCCCCCCEEEECCCCCCCCEEEEECCCCCCCCEEEEECC',
 'seq': 'DMGCVASWSGKELKCGSGIFVVDNVHTWTEQYKFQPESPARLASAILNAHKDGVCGIRSTTRLENVMWKQITNELNYVLWEGGHDLTVVAGDVKGVLTKGKRALTPPVSDLKYSWKTWGKAKIFTPEARNSTFLIDGPDTSECPNERRAWNSLEVEDYGFGMFTTNIWMKFREGSSEVCDHRLMSAAIKDQKAVHADMGYWIESSKNQTWQIEKASLIEVKTCLWPKTHTLWSNGVLESQMLIPKSYAGPFSQHNYRQGYA

In [96]:
def avaliar_epitopos_com_horiz(df_epitopos, conf, pred, seq):
    """Cruza o DataFrame de epítopos com as strings de predição do .horiz já processadas."""
    resultados = []
    
    for idx, row in df_epitopos.iterrows():
        # Identifica a coluna correta do peptídeo dependendo do estágio do pipeline
        col_pep = 'peptide' if 'peptide' in df_epitopos.columns else 'Epitope sequence'
        peptideo = row[col_pep]
        start = int(row['start'])
        end = int(row['end'])
        
        # Ajuste para índice em Python (base 0)
        idx_inicio = start - 1
        idx_fim = end
        
        # Extrai os trechos correspondentes nas strings do horiz
        sub_pred = pred[idx_inicio:idx_fim]
        sub_conf = conf[idx_inicio:idx_fim]
        
        # Conta a proporção de cada estrutura secundária
        # 'H' = Hélice, 'E' = Fita Beta, 'C' ou '.' = Coil/Alça
        n_coils = sub_pred.count('C') + sub_pred.count('.')
        n_helix = sub_pred.count('H')
        n_strand = sub_pred.count('E')
        
        resultados.append({
            'Epitope': peptideo,
            'Start': start,
            'End': end,
            'Pred_SS': sub_pred,
            'Coils': n_coils,
            'Helices': n_helix,
            'Strands': n_strand
        })
        
    return pd.DataFrame(resultados)

In [97]:
conf = denv_horiz["conf"]
pred = denv_horiz["pred"]
seq = denv_horiz["seq"]

avaliacao_epitopos = avaliar_epitopos_com_horiz(conservacy_result_filtered, conf, pred, seq)
avaliacao_epitopos

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands
0,HTWTEQYKF,26,34,CCCEEECEE,4,0,5


In [98]:
# Calcula o percentual de Coils (flexibilidade) para cada epítopo
avaliacao_epitopos['Percent_Coils'] = (
    avaliacao_epitopos['Coils'] / (avaliacao_epitopos['Coils'] + avaliacao_epitopos['Helices'] + avaliacao_epitopos['Strands'])
) * 100

# Ordena os epítopos do mais flexível (maior percentual de coils) para o mais rígido
epitopos_flexiveis = avaliacao_epitopos.sort_values(
    by=['Percent_Coils', 'Coils'], 
    ascending=[False, False]
).reset_index(drop=True)

epitopos_flexiveis

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils
0,HTWTEQYKF,26,34,CCCEEECEE,4,0,5,44.444444


In [99]:
# Calcula o percentual de Coils (flexibilidade) para cada epítopo
avaliacao_epitopos['Percent_Coils'] = (
    avaliacao_epitopos['Coils'] / (avaliacao_epitopos['Coils'] + avaliacao_epitopos['Helices'] + avaliacao_epitopos['Strands'])
) * 100

# Ordena os epítopos do mais flexível (maior percentual de coils) para o mais rígido
epitopos_flexiveis = avaliacao_epitopos.sort_values(
    by=['Percent_Coils', 'Coils'], 
    ascending=[False, False]
).reset_index(drop=True)

epitopos_flexiveis

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils
0,HTWTEQYKF,26,34,CCCEEECEE,4,0,5,44.444444


#### Filtrar epítopos que tenham pela % de sua estrutura composta por Coils

In [100]:
valor_corte_flexibilidade = parametros.corte_flexibilidade
print(f"Valor de corte de flexibilidade (Percent_Coils): {valor_corte_flexibilidade}%")

Valor de corte de flexibilidade (Percent_Coils): 35.0%


In [101]:
corte_flexibilidade = valor_corte_flexibilidade
epitopos_filtrados_flex = epitopos_flexiveis[epitopos_flexiveis['Percent_Coils'] >= corte_flexibilidade]

epitopos_filtrados_flex

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils
0,HTWTEQYKF,26,34,CCCEEECEE,4,0,5,44.444444


### Merge de dataframes

In [102]:
# Seleciona apenas as colunas exclusivas de conservação (excluindo 'start' e 'end' repetidos, se houver)
colunas_para_pegar = [c for c in conservacy_result_filtered.columns if c not in ['start', 'end', 'Epitope sequence']]
colunas_para_pegar.append('Epitope sequence') # Garante que a chave está incluída

# Realiza o merge limpo
df_consolidado = pd.merge(
    epitopos_filtrados_flex,
    conservacy_result_filtered[colunas_para_pegar],
    left_on='Epitope',
    right_on='Epitope sequence',
    how='inner'
)

# Remove se sobrou a coluna 'Epitope sequence' duplicada:
if 'Epitope sequence' in df_consolidado.columns and 'Epitope' in df_consolidado.columns:
    df_consolidado = df_consolidado.drop(columns=['Epitope sequence'])

df_consolidado

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils,Epitope name,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,alelos,percent_match
0,HTWTEQYKF,26,34,CCCEEECEE,4,0,5,44.444444,NP 2,9,98.66% (147/149),88.89%,100.00%,10,0.28,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2...",98.66


In [103]:
epitopos_filtrados_flex.to_csv("epitopos_validados_flexibilidade_tcell1.csv", index=False)